In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import InvalidSessionIdException, TimeoutException, WebDriverException
import time
import os
import shutil
import pandas as pd

# Load TDR list
df = pd.read_excel("Maharashtra_filtered_1.xlsx")
tdr_list = df["TDR"].astype(str).tolist()

# Setup download directory
base_download_dir = os.path.join(os.getcwd(), "BOQ_Downloads")
os.makedirs(base_download_dir, exist_ok=True)

# Chrome options
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": base_download_dir,
    "download.prompt_for_download": False,
    "profile.default_content_settings.popups": 0,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

def wait_for(driver, by, selector, timeout=15):
    return WebDriverWait(driver, timeout).until(EC.presence_of_element_located((by, selector)))

def wait_for_download_complete(folder, timeout=15):
    end_time = time.time() + timeout
    while time.time() < end_time:
        files = os.listdir(folder)
        if any(f.endswith((".xls", ".xlsx")) for f in files) and not any(f.endswith(".crdownload") for f in files):
            return True
        time.sleep(1)
    return False

def start_driver(prompt_login=True):
    driver = webdriver.Chrome(options=options)
    driver.get("https://www.tenderdetail.com/Account/LogOn")
    if prompt_login:
        input("🔐 Please log in manually, then press ENTER to continue after dashboard fully loads...")
    return driver

driver = start_driver(prompt_login=True)
summary = []

for tdr in tdr_list:
    print(f"\n🔎 Searching for TDR: {tdr}")
    tdr_dir = os.path.join(base_download_dir, tdr)
    os.makedirs(tdr_dir, exist_ok=True)

    try:
        driver.get(f"https://www.tenderdetail.com/registeruser/searchtenders?ServiceType=1&nm={tdr}")
        wait_for(driver, By.CSS_SELECTOR, "#tenderbreif > span")
    except (InvalidSessionIdException, WebDriverException):
        print(f"❌ Session expired. Restarting driver...")
        try:
            driver.quit()
        except:
            pass
        driver = start_driver(prompt_login=True)
        try:
            driver.get(f"https://www.tenderdetail.com/registeruser/searchtenders?ServiceType=1&nm={tdr}")
            wait_for(driver, By.CSS_SELECTOR, "#tenderbreif > span")
        except Exception as e:
            print(f"❌ Failed after restart: {e}")
            summary.append({"TDR": tdr, "BOQ Files": "Session Error", "NIT Files": "Session Error"})
            continue
    except Exception as e:
        print(f"❌ Error loading page: {e}")
        summary.append({"TDR": tdr, "BOQ Files": "Page Load Error", "NIT Files": "Page Load Error"})
        continue

    try:
        brief = driver.find_element(By.CSS_SELECTOR, "#tenderbreif > span").text
        print("🔍 Tender brief:", brief)
        if tdr not in brief:
            summary.append({"TDR": tdr, "BOQ Files": "", "NIT Files": ""})
            continue
    except:
        summary.append({"TDR": tdr, "BOQ Files": "", "NIT Files": ""})
        continue

    try:
        view_btns = driver.find_elements(By.CSS_SELECTOR, ".viewnotice")
        if not view_btns:
            summary.append({"TDR": tdr, "BOQ Files": "No View Notice", "NIT Files": ""})
            continue
        driver.execute_script("arguments[0].click();", view_btns[0])
        wait_for(driver, By.TAG_NAME, "a")
        time.sleep(2)
    except:
        summary.append({"TDR": tdr, "BOQ Files": "View Notice Failed", "NIT Files": ""})
        continue

    boq_files = []
    nit_files = []

    try:
        links = driver.find_elements(By.TAG_NAME, "a")
        hrefs = [link.get_attribute("href") for link in links if link.get_attribute("href") and "FileName=" in link.get_attribute("href")]
    except Exception as e:
        print(f"⚠️ Link collection failed: {e}")
        summary.append({"TDR": tdr, "BOQ Files": "No Links", "NIT Files": "No Links"})
        continue

    for href in hrefs:
        filename = href.split("FileName=")[-1]

        if filename.lower().endswith(".xls") and "boq" in filename.lower():
            print(f"📥 Downloading BOQ: {filename}")
            try:
                driver.get(href)
                if wait_for_download_complete(base_download_dir, timeout=10):
                    boq_files.append(filename)
                time.sleep(2)
            except Exception as e:
                print(f"❌ BOQ download failed: {e}")

        elif filename.lower().endswith(".pdf") and "nit" in filename.lower():
            print(f"📥 Downloading NIT: {filename}")
            try:
                driver.get(href)
                time.sleep(3)
                if any(f.endswith(".pdf") and "nit" in f.lower() for f in os.listdir(base_download_dir)):
                    nit_files.append(filename)
            except Exception as e:
                print(f"❌ NIT download failed: {e}")

    try:
        for f in os.listdir(base_download_dir):
            if f.lower().endswith((".xls", ".xlsx", ".pdf")):
                src = os.path.join(base_download_dir, f)
                dst = os.path.join(tdr_dir, f)
                shutil.move(src, dst)
    except Exception as e:
        print(f"⚠️ File move error: {e}")

    summary.append({
        "TDR": tdr,
        "BOQ Files": ", ".join(boq_files) if boq_files else "Not Found",
        "NIT Files": ", ".join(nit_files) if nit_files else "Not Found"
    })

# Save summary
summary_path = os.path.join(base_download_dir, "summary.xlsx")
pd.DataFrame(summary).to_excel(summary_path, index=False)
print(f"\n✅ Completed. Summary saved at: {summary_path}")

driver.quit()


🔐 Please log in manually, then press ENTER to continue after dashboard fully loads... 



🔎 Searching for TDR: 49453675
🔍 Tender brief: TDR : 49453675 Corrigendum : tender for supply of ventilator tubing.
📥 Downloading BOQ: BOQ_2057455.xls

🔎 Searching for TDR: 49453586
🔍 Tender brief: TDR : 49453586 tender for supply of water supply repair and related material , pvc pipe 1/2 inch , pvc pipe 3/4 inch , pvc pipe 1 inch , pvc pipe 1 1/4 inch , pvc pipe 1 1/2 inch , pvc pipe 2 inch , pvc pipe 2 1/2 inch supreme , pvc pipe 3 1/2 inch supreme , pvc pipe 4
📥 Downloading BOQ: BOQ_2057585.xls

🔎 Searching for TDR: 49453585
🔍 Tender brief: TDR : 49453585 supply of precast concrete pipe np2 150 mm dia 2 mtr , precast concrete pipe np2 200 mm dia 2 mtr , precast concrete pipe np2 250 mm dia 2 mtr , precast concrete pipe np2 300 mm dia 2 mtr , precast concrete pipe np2 450 mm dia 2 mtr , precast concrete pip
📥 Downloading BOQ: BOQ_2057568.xls

🔎 Searching for TDR: 49452716
🔍 Tender brief: TDR : 49452716 bids are invited for hermatic compressor, type reciprocating for 5tr chilled water